In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

ratings = pd.read_csv('../data/ratings.csv')
books = pd.read_csv('../data/books.csv')

user_counts = ratings['user_id'].value_counts()
active_users = user_counts[user_counts>=5].index
ratings = ratings[ratings['user_id'].isin(active_users)]

user_ids = ratings['user_id'].unique()
book_ids = ratings['book_id'].unique()
user2idx = {user_id: idx for idx, user_id in enumerate(user_ids)}
book2idx = {book_id: idx for idx, book_id in enumerate(book_ids)}
ratings['user_idx'] = ratings['user_id'].map(user2idx)
ratings['book_idx'] = ratings['book_id'].map(book2idx)

n_users=len(user2idx)
n_books=len(book2idx)

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

print(f"Users: {n_users:,} | Books: {n_books:,}")
print(f"Train: {len(train_data):,} | Test: {len(test_data):,}")

In [ ]:
print("Loading MiniLM model...")
encoder = SentenceTransformer('all-MiniLM-L6-v2')

books['description']=books['title'].fillna('')+'by'+books['authors'].fillna('')

print("Generating book embeddings for all 10.000 books.....")
descriptions = books['description'].tolist()
book_embeddings = encoder.encode(descriptions, batch_size=64, show_progress_bar=True)

print(f"\nEmbedding shape: {book_embeddings.shape}")
print(f"Each book is represented as {book_embeddings.shape[1]}-dimensional vector")

np.save('../data/book_embeddings.npy', book_embeddings)
print("Embeddings saved to ../data/book_embeddings.npy")

In [ ]:
book_embeddings = np.load('../data/book_embeddings.npy')
print(f"Embeddings loaded: {book_embeddings.shape}")

idx2bookid = {v: k for k, v in book2idx.items()}
bookid2row = {row['id']: idx for idx, row in books.iterrows()}

embedding_dim = 50
content_dim = book_embeddings.shape[1]

print(f"Collaborative embedding dim: {embedding_dim}")
print(f"Content embedding dim: {content_dim}")
print(f"Total input to dense layers: {embedding_dim * 2 + content_dim}")

In [ ]:
# class HybridDataset(Dataset):
#     def __init__(self, df, book_embeddings, bookid2row, idx2bookid):
#         self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
#         self.books = torch.tensor(df['book_idx'].values, dtype=torch.long)
#         self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)

#         content_embs = []
#         for book_idx in df['book_idx'].values:
#             book_id = idx2bookid[book_idx]
#             row = bookid2row.get(book_id, 0)
#             content_embs.append(book_embeddings[row])

#             self.content_embeddings = torch.tensor(
#                 np.array(content_embs), dtype=torch.float32
#             )

#         def __len__(self):
#             return len(self.ratings)
#         def __getitem__(self,idx):
#             return self.users[idx], self.books[idx], self.content_embeddings[idx], self.ratings[idx]

# print("Building hybrid datasets...")
# train_dataset = HybridDataset(train_data, book_embeddings, bookid2row, idx2bookid)
# test_dataset = HybridDataset(test_data, book_embeddings, bookid2row, idx2bookid)

# train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
# test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

# print(f"Train batches: {len(train_loader)}")
# print(f"Test batches: {len(test_loader)}")

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, df, book_embeddings, bookid2row, idx2bookid):
        self.users = torch.tensor(df['user_idx'].values, dtype=torch.long)
        self.books = torch.tensor(df['book_idx'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        
        # Vectorised lookup — much faster than looping
        book_idx_array = df['book_idx'].values
        book_id_array = np.array([idx2bookid[idx] for idx in book_idx_array])
        row_array = np.array([bookid2row.get(bid, 0) for bid in book_id_array])
        self.content_embeddings = torch.tensor(
            book_embeddings[row_array], dtype=torch.float32
        )
    
    def __len__(self):
        return len(self.ratings)
    
    def __getitem__(self, idx):
        return self.users[idx], self.books[idx], self.content_embeddings[idx], self.ratings[idx]

print("Building hybrid datasets...")
train_dataset = HybridDataset(train_data, book_embeddings, bookid2row, idx2bookid)
test_dataset = HybridDataset(test_data, book_embeddings, bookid2row, idx2bookid)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, n_users, n_books, embedding_dim=50, content_dim=384, hidden_layers=[256, 128, 64], dropout=0.2):
        super().__init__()
        
        self.user_embeddings = nn.Embedding(n_users, embedding_dim)
        self.book_embeddings = nn.Embedding(n_books, embedding_dim)
        
        nn.init.normal_(self.user_embeddings.weight, std=0.01)
        nn.init.normal_(self.book_embeddings.weight, std=0.01)
        
        input_dim = embedding_dim * 2 + content_dim
        
        layers = []
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim
        
        layers.append(nn.Linear(input_dim, 1))
        self.network = nn.Sequential(*layers)
    
    def forward(self, user_idx, book_idx, content_emb):
        user_emb = self.user_embeddings(user_idx)
        book_emb = self.book_embeddings(book_idx)
        concat = torch.cat([user_emb, book_emb, content_emb], dim=1)
        return self.network(concat).squeeze()

model = HybridModel(n_users, n_books, embedding_dim=50, content_dim=384)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for user_idx, book_idx, content_emb, rating in loader:
        optimizer.zero_grad()
        prediction = model(user_idx, book_idx, content_emb)
        loss = criterion(prediction, rating)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for user_idx, book_idx, content_emb, rating in loader:
            prediction = model(user_idx, book_idx, content_emb)
            loss = criterion(prediction, rating)
            total_loss += loss.item()
    return total_loss / len(loader)

n_epochs = 20
patience = 3
best_test_loss = float('inf')
epochs_without_improvement = 0
train_losses, test_losses = [], []
best_model_state = None

for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    test_loss = evaluate(model, test_loader, criterion)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    if test_loss < best_test_loss:
        best_test_loss = test_loss
        best_model_state = model.state_dict().copy()
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    
    print(f"Epoch {epoch+1:02d}/{n_epochs} | Train: {train_loss:.4f} | Test: {test_loss:.4f} | Best: {best_test_loss:.4f} {'*' if epochs_without_improvement == 0 else ''}")
    
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print(f"\nBest test loss: {best_test_loss:.4f}")
print(f"Best RMSE: {best_test_loss**0.5:.4f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, len(test_losses)+1), test_losses, label='Test Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Hybrid Model Training Curves')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

import os
torch.save({
    'model_state_dict': best_model_state,
    'user2idx': user2idx,
    'book2idx': book2idx,
    'n_users': n_users,
    'n_books': n_books,
    'embedding_dim': 50,
    'content_dim': 384,
    'hidden_layers': [256, 128, 64],
    'dropout': 0.2,
    'best_test_loss': best_test_loss,
}, '../models/hybrid_model.pt')

print(f"Model saved.")
print(f"File size: {os.path.getsize('../models/hybrid_model.pt') / 1024 / 1024:.1f} MB")

# Hybrid Model Summary

## Architecture
- User embeddings: 35,710 × 50
- Book embeddings: 10,000 × 50
- MiniLM content embeddings: 10,000 × 384
- Concat (484d) → Linear(256) → ReLU → Dropout(0.2)
- → Linear(128) → ReLU → Dropout(0.2)
- → Linear(64) → ReLU → Dropout(0.2)
- → Linear(1)
- Total parameters: 2,450,877

## Training
- Optimiser: Adam (lr=0.001)
- Loss: MSE
- Max epochs: 20, batch size: 1024
- Early stopping patience: 3
- Stopped at epoch 8, best at epoch 5

## Final Model Comparison
| Model | RMSE |
|---|---|
| Item-Item CF | 0.9069 |
| Matrix Factorisation | 0.832 |
| NCF | 0.836 |
| Hybrid + MiniLM | 0.831 |

## Key Learnings
- Content embeddings add genuine signal, best RMSE across all models
- Cold start solved, new books use MiniLM embeddings immediately
- Vectorised NumPy lookup essential for large dataset performance
- Early stopping caught overfitting at epoch 8

## Next Steps
- Upload model weights to Hugging Face Hub
- Build FastAPI backend
- Build Next.js frontend